# Deriving the slope and the intercept

The residual sum of squares is a function of two variables. Setting both partial derivatives equal to zero produces the normal equations. Solving those equations produces the slope and intercept that every later lesson will use.


## The normal equations

Let

$$
S(b_0, b_1) = \sum_{i=1}^{n}\bigl(y_i - b_0 - b_1 x_i\bigr)^2.
$$

Differentiate under the finite sum:

$$
\begin{aligned}
\dfrac{\partial S}{\partial b_0}
&= -2\sum_{i=1}^{n}\bigl(y_i - b_0 - b_1 x_i\bigr), \\
\dfrac{\partial S}{\partial b_1}
&= -2\sum_{i=1}^{n} x_i\bigl(y_i - b_0 - b_1 x_i\bigr).
\end{aligned}
$$

Set both equal to zero and divide by $-2$. Writing $\sum$ for a sum over the $n$ observations,

$$
\begin{aligned}
n b_0 + b_1 \sum x_i &= \sum y_i, \\
b_0 \sum x_i + b_1 \sum x_i^2 &= \sum x_i y_i.
\end{aligned}
$$

These are the normal equations of simple regression. The first says that the residuals sum to zero. The second says that the residuals are orthogonal to the predictor $x$.

To see that the critical point is a minimum, form the second derivatives:

$$
\dfrac{\partial^2 S}{\partial b_0^2} = 2n, \qquad
\dfrac{\partial^2 S}{\partial b_1^2} = 2\sum x_i^2, \qquad
\dfrac{\partial^2 S}{\partial b_0\,\partial b_1} = 2\sum x_i.
$$

The Hessian has determinant $4\bigl(n\sum x_i^2 - (\sum x_i)^2\bigr) = 4n\sum (x_i - \bar x)^2$. Whenever $x$ is not constant, that determinant is positive and the first entry $2n$ is positive, so the Hessian is positive definite and the critical point is the unique minimum.


## The closed form

Divide the first normal equation by $n$:

$$
b_0 = \bar y - b_1 \bar x.
$$

Substitute this expression into the second, or equivalently center the data first. With the centered sums

$$
S_{xx} = \sum_{i=1}^{n}(x_i - \bar x)^2, \qquad
S_{xy} = \sum_{i=1}^{n}(x_i - \bar x)(y_i - \bar y),
$$

the slope collapses to one division,

$$
b_1 = \dfrac{S_{xy}}{S_{xx}}, \qquad b_0 = \bar y - b_1 \bar x,
$$

provided $S_{xx} > 0$. The same slope can be written without explicitly centering:

$$
b_1 = \dfrac{n\sum x_i y_i - \bigl(\sum x_i\bigr)\bigl(\sum y_i\bigr)}{n\sum x_i^2 - \bigl(\sum x_i\bigr)^2}.
$$

The two fractions are equal because centering removes the same cross term from the numerator and the denominator.


## The three-point table, by hand

Use $(x, y) = (1, 2),\ (2, 3),\ (3, 5)$.

$$
\begin{aligned}
n &= 3, \\
\sum x &= 6, \qquad \sum y = 10, \\
\sum xy &= 1\cdot 2 + 2\cdot 3 + 3\cdot 5 = 23, \\
\sum x^2 &= 1 + 4 + 9 = 14.
\end{aligned}
$$

The uncentered formula gives

$$
b_1 = \dfrac{3\cdot 23 - 6\cdot 10}{3\cdot 14 - 6^2} = \dfrac{69 - 60}{42 - 36} = \dfrac{9}{6} = \dfrac{3}{2}.
$$

Then

$$
b_0 = \dfrac{10 - \tfrac{3}{2}\cdot 6}{3} = \dfrac{10 - 9}{3} = \dfrac{1}{3}.
$$

The centered route must agree. $\bar x = 2$ and $\bar y = 10/3$, so

$$
\begin{aligned}
S_{xx} &= (-1)^2 + 0^2 + 1^2 = 2, \\
S_{xy} &= (-1)\left(2 - \dfrac{10}{3}\right) + 0 + (1)\left(5 - \dfrac{10}{3}\right)
= \dfrac{4}{3} + \dfrac{5}{3} = 3, \\
b_1 &= \dfrac{3}{2}, \qquad
b_0 = \dfrac{10}{3} - \dfrac{3}{2}\cdot 2 = \dfrac{1}{3}.
\end{aligned}
$$

The fitted line is $\widehat y = \dfrac{1}{3} + \dfrac{3}{2}x$. Its cost is computed in the next section; the derivation here only identifies the coefficients.


In [1]:
# Both closed forms, then SymPy's derivatives, on the three-point table.
from fractions import Fraction
import sympy as sp

x = [Fraction(1), Fraction(2), Fraction(3)]
y = [Fraction(2), Fraction(3), Fraction(5)]
n = len(x)
sum_x, sum_y = sum(x), sum(y)
sum_xy = sum(a * b for a, b in zip(x, y))
sum_x2 = sum(a * a for a in x)
slope_sums = (n * sum_xy - sum_x * sum_y) / (n * sum_x2 - sum_x ** 2)
intercept_sums = (sum_y - slope_sums * sum_x) / n
xbar, ybar = sum_x / n, sum_y / n
sxx = sum((a - xbar) ** 2 for a in x)
sxy = sum((a - xbar) * (b - ybar) for a, b in zip(x, y))
slope_centered = sxy / sxx
intercept_centered = ybar - slope_centered * xbar
print("from sums", intercept_sums, slope_sums)
print("from centered sums", intercept_centered, slope_centered)

b0, b1 = sp.symbols("b0 b1")
loss = sum((yy - b0 - b1 * xx) ** 2 for xx, yy in zip(x, y))
solution = sp.solve([sp.diff(loss, b0), sp.diff(loss, b1)], [b0, b1])
print("from the partial derivatives", solution[b0], solution[b1])
assert slope_sums == slope_centered == solution[b1] == Fraction(3, 2)
assert intercept_sums == intercept_centered == solution[b0] == Fraction(1, 3)


from sums 1/3 3/2
from centered sums 1/3 3/2
from the partial derivatives 1/3 3/2


## The cost of the winning line

The fitted values and residuals are

$$
\begin{aligned}
\widehat y &= \dfrac{1}{3} + \dfrac{3}{2},\ 
\dfrac{1}{3} + 3,\ 
\dfrac{1}{3} + \dfrac{9}{2}
= \dfrac{11}{6},\ \dfrac{10}{3},\ \dfrac{29}{6}, \\
e &= 2 - \dfrac{11}{6},\ 
3 - \dfrac{10}{3},\ 
5 - \dfrac{29}{6}
= \dfrac{1}{6},\ -\dfrac{1}{3},\ \dfrac{1}{6}.
\end{aligned}
$$

Two checks required by the normal equations:

$$
\sum e_i = \dfrac{1}{6} - \dfrac{2}{6} + \dfrac{1}{6} = 0, \qquad
\sum x_i e_i = \dfrac{1}{6} - \dfrac{2}{3} + \dfrac{1}{2} = 0.
$$

And the cost, which beat the candidate cost $1$ from the previous lesson, is

$$
\mathrm{RSS} = \left(\dfrac{1}{6}\right)^2 + \left(\dfrac{1}{3}\right)^2 + \left(\dfrac{1}{6}\right)^2 = \dfrac{1 + 4 + 1}{36} = \dfrac{1}{6}.
$$


In [2]:
# Fitted values, both orthogonality checks, and the residual sum of squares.
from fractions import Fraction
x = [Fraction(1), Fraction(2), Fraction(3)]
y = [Fraction(2), Fraction(3), Fraction(5)]
intercept, slope = Fraction(1, 3), Fraction(3, 2)
fitted = [intercept + slope * xi for xi in x]
errors = [yi - yh for yi, yh in zip(y, fitted)]
print("fitted", fitted)
print("residuals", errors)
print("sum of residuals", sum(errors))
print("sum of x times residuals", sum(xi * ei for xi, ei in zip(x, errors)))
print("RSS", sum(ei ** 2 for ei in errors))
assert fitted == [Fraction(11, 6), Fraction(10, 3), Fraction(29, 6)]
assert errors == [Fraction(1, 6), Fraction(-1, 3), Fraction(1, 6)]
assert sum(errors) == 0
assert sum(xi * ei for xi, ei in zip(x, errors)) == 0
assert sum(ei ** 2 for ei in errors) == Fraction(1, 6)


fitted [Fraction(11, 6), Fraction(10, 3), Fraction(29, 6)]
residuals [Fraction(1, 6), Fraction(-1, 3), Fraction(1, 6)]
sum of residuals 0
sum of x times residuals 0
RSS 1/6


## Pitfall

If every $x_i$ is the same number, then $S_{xx} = 0$ and the slope formula divides by zero. The data no longer identify a slope: any steep line can be made to pass through the one vertical stack by moving the intercept, and the algebra correctly refuses to choose one. On $x = (2, 2, 2)$ and $y = (2, 3, 5)$, the only determined fact is the best constant, $\bar y = 10/3$.


In [3]:
# A constant predictor has no slope to estimate.
from fractions import Fraction
x = [Fraction(2), Fraction(2), Fraction(2)]
y = [Fraction(2), Fraction(3), Fraction(5)]
xbar = sum(x) / len(x)
sxx = sum((a - xbar) ** 2 for a in x)
print("Sxx", sxx)
print("best constant", sum(y) / len(y))
assert sxx == 0
try:
    print((1) / sxx)
    raise AssertionError("division by zero was expected")
except ZeroDivisionError:
    print("slope formula refused: ZeroDivisionError")


Sxx 0
best constant 10/3
slope formula refused: ZeroDivisionError


## Summary

- The least-squares line is the unique minimum of $S(b_0, b_1)$ when $x$ is not constant.
- Its residuals sum to zero and are orthogonal to $x$. Those two sentences are the normal equations.
- $b_1 = S_{xy}/S_{xx}$ and $b_0 = \bar y - b_1\bar x$. On the three-point table, the line is $\tfrac{1}{3} + \tfrac{3}{2}x$ and $\mathrm{RSS} = \tfrac{1}{6}$.
